# 05 - BASELINE: Punto de referencia / Reference point

## SPA

**Propósito**
Establecer el punto de referencia contra el que se medirá todo lo que venga después. El objetivo no es conseguir un buen modelo sino tener una base con la que comparar después.

**Entrada**
`telco_churn.gold.customer_features` (7043 filas x 28 columnas)

**Salida**
Ninguna tabla. Este notebook no persiste nada. El modelo tampoco se guarda.

**Decisiones de diseño**

El reparto entre entrenamiento y prueba usa el 20 % para prueba, estratificado por la variable objetivo para que el 26,5 % de bajas se mantenga en ambas mitades y con la random seed en 42 para que el resultado sea reproducible.

Se entrenan dos modelos. El primero es un clasificador "tonto" que predice siempre la clase mayoritaria y sirve como la base más baja. El segundo es una regresión logística sin ajustar ningún hiperparámetro.

Todo el preprocesado vive dentro de un `Pipeline` de scikit-learn junto al modelo. Así el escalado y la codificación se ajustan únicamente con los datos de entrenamiento, sin posibilidad de fuga. El objeto resultante sabe ir de datos crudos a predicción por sí solo. Eso es lo que se registrará y se desplegará más adelante.

Las columnas se reparten en tres grupos con tratamiento distinto. Las quince categóricas de texto se convierten en indicadoras con `handle_unknown="ignore"`, para que un valor desconocido en producción no rompa el endpoint. Las seis numéricas se escalan. Las cinco que ya vienen en 0 y 1 pasan sin tocar.

---

## Conclusiones

### Los números de referencia

| Modelo | Accuracy | Precisión (baja) | Recall (baja) | F1 (baja) |
|---|---|---|---|---|
| Clasificador dummy | 0,735 | 0,00 | 0,00 | 0,00 |
| Regresión logística | 0,798 | 0,65 | 0,53 | 0,58 |

### Qué significan en clientes

De los 374 clientes del conjunto de prueba que se dieron de baja, el modelo detecta unos 198 y se le escapan unos 176. Para conseguirlo marca a unos 305 clientes en total, de los cuales 107 no se iban a ir y habrían recibido una oferta de retención innecesaria.

### Por qué la accuracy no sirve aquí

El clasificador dummy acierta el 73,5 % de las veces sin mirar un solo dato, simplemente prediciendo que nadie se va. Es la base que cualquier modelo tiene que superar para justificar su existencia.

La regresión logística sube la accuracy hasta el 79,8 %, apenas seis puntos. Pero el recall de la clase minoritaria pasa de 0 a 0,53, de no detectar absolutamente a nadie a detectar a la mitad. La accuracy se mueve seis puntos para describir un cambio de inútil a funcional.

En el informe de clasificación pasa lo mismo con los promedios. El promedio ponderado da 0,73 porque la clase mayoritaria lo arrastra, mientras que el promedio macro, que trata ambas clases por igual, da 0,50 y dice la verdad.

### El modelo confirma el EDA

Los coeficientes reproducen los hallazgos de la Fase 2. La antigüedad y el contrato de dos años son las señales más fuertes hacia la permanencia, el contrato mensual y la fibra empujan hacia la baja y los servicios de soporte y seguridad retienen. Es una validación cruzada de todo el trabajo anterior.

Aparecen dos cosas dignas de mención. La variable `is_new_customer` sale con signo negativo, indicando que los clientes de antigüedad cero tienden a quedarse. No es un error sino la tautología que ya anticipamos al crearla, ya que un cliente dado de alta este mes no ha tenido ocasión de darse de baja. Con once casos de 5634 apenas influye.

Y faltan del top 15 algunas variables que el análisis exploratorio señalaba con fuerza, como el cheque electrónico o `fiber_no_support`. No es que no importen. Su señal está repartida entre varias columnas que dicen casi lo mismo, de modo que ninguna destaca por separado.

# 05 - BASELINE: Punto de referencia / Reference point

## ENG

**Purpose**
Establish the reference point against which everything that follows will be measured. The goal is not to obtain a good model but to have a baseline to compare against later.

**Input**
`telco_churn.gold.customer_features` (7,043 rows x 28 columns)

**Output**
No table. This notebook does not persist anything. The model is not saved either.

**Design decisions**

The train and test split uses 20 % for testing, stratified by the target variable so that the 26.5 % churn rate is preserved in both halves, and with the random seed set to 42 so the result is reproducible.

Two models are trained. The first is a dummy classifier that always predicts the majority class and serves as the lowest possible baseline. The second is a logistic regression with no hyperparameter tuning.

All preprocessing lives inside a scikit-learn `Pipeline` alongside the model. That way scaling and encoding are fitted on the training data only, with no possibility of leakage. The resulting object knows how to go from raw data to prediction on its own. That is what will be registered and deployed later on.

Columns are split into three groups with different treatment. The fifteen text categoricals are turned into indicator variables with `handle_unknown="ignore"`, so that an unseen value in production does not break the endpoint. The six numeric ones are scaled. The five that already come as 0 and 1 are passed through untouched.

---

## Conclusions

### The reference numbers

| Model | Accuracy | Precision (churn) | Recall (churn) | F1 (churn) |
|---|---|---|---|---|
| Dummy classifier | 0.735 | 0.00 | 0.00 | 0.00 |
| Logistic regression | 0.798 | 0.65 | 0.53 | 0.58 |

### What they mean in customers

Of the 374 customers in the test set who churned, the model catches around 198 and misses around 176. To do so it flags roughly 305 customers in total, of which 107 were never going to leave and would have received an unnecessary retention offer.

### Why accuracy does not work here

The dummy classifier is right 73.5 % of the time without looking at a single data point, simply by predicting that nobody leaves. That is the baseline any model has to beat in order to justify its existence.

Logistic regression lifts accuracy to 79.8 %, barely six points. But recall on the minority class goes from 0 to 0.53, from detecting absolutely nobody to detecting half of them. Accuracy moves six points to describe a change from useless to functional.

The same thing happens with the averages in the classification report. The weighted average gives 0.73 because the majority class drags it up, while the macro average, which treats both classes equally, gives 0.50 and tells the truth.

### The model confirms the EDA

The coefficients reproduce the findings from Phase 2. Tenure and the two-year contract are the strongest signals towards staying, month-to-month contracts and fibre push towards leaving, and support and security services retain customers. It is a cross-validation of all the previous work.

Two things are worth noting. The `is_new_customer` variable comes out with a negative sign, indicating that customers with zero tenure tend to stay. This is not an error but the tautology anticipated when the feature was created, since a customer signed up this month has had no chance to leave. With eleven cases out of 5,634 it barely has any influence.

Some variables the exploratory analysis flagged strongly are missing from the top 15, such as electronic check or `fiber_no_support`. It is not that they do not matter. Their signal is split across several columns saying almost the same thing, so none of them stands out on its own.

In [0]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from churn.validation import validar_gold

from churn.config import (GOLD_TABLE, ID, TARGET, RANDOM_SEED)
from churn.pipeline import construir_pipeline

from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

In [0]:
pdf = spark.table(GOLD_TABLE).toPandas()
validar_gold(pdf)

In [0]:
X = pdf.drop(columns=[ID, TARGET])
y = pdf[TARGET]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_SEED, stratify=y)

In [0]:
X_train.size

In [0]:
y_train.size

In [0]:
dummy_clf = DummyClassifier(strategy="most_frequent")
dummy_clf.fit(X_train, y_train)

In [0]:
y_pred = dummy_clf.predict(X_test)

In [0]:
print(f"Accuracy: {accuracy_score(y_test, y_pred):.3f}")
print(classification_report(y_test, y_pred))

In [0]:
pipe = construir_pipeline(LogisticRegression(max_iter=1000, random_state=RANDOM_SEED))
pipe.fit(X_train, y_train)

In [0]:
y_pred = pipe.predict(X_test)

print(f"Accuracy: {accuracy_score(y_test, y_pred):.3f}")
print(classification_report(y_test, y_pred))

In [0]:
ConfusionMatrixDisplay.from_estimator(
    pipe, X_test, y_test,
    display_labels=["Se queda", "Se va"],
    cmap="Blues",
)

In [0]:
nombres = pipe.named_steps["preprocesado"].get_feature_names_out()
coefs = pipe.named_steps["modelo"].coef_[0]

importancia = pd.DataFrame({"variable": nombres, "coeficiente": coefs})
importancia["magnitud"] = importancia["coeficiente"].abs()
importancia = importancia.sort_values("magnitud", ascending=False)

display(importancia.head(15))

In [0]:
top = importancia.head(15).sort_values("coeficiente")
colores = ["tab:red" if c > 0 else "tab:blue" for c in top["coeficiente"]]

plt.figure(figsize=(9, 6))
plt.barh(top["variable"], top["coeficiente"], color=colores)
plt.axvline(0, color="black", linewidth=0.8)
plt.xlabel("Coeficiente  (rojo = empuja a la baja, azul = retiene)")
plt.title("Variables más influyentes — regresión logística")
plt.tight_layout();